# Lesson 05 - Agentic RAG

## Setup

This notebook demonstrates the Agentic RAG (Retrieval-Augmented Generation) pattern using the Microsoft Agent Framework and Azure AI Search hybrid retrieval.

**Prerequisites:**
- `AZURE_SEARCH_SERVICE_ENDPOINT` — your Azure AI Search service endpoint, for example `https://azure-search-service-01.search.windows.net`
- `AZURE_SEARCH_INDEX_NAME` — your Azure AI Search index name, for example `demo-datasource-ks-index`
- `AZURE_SEARCH_API_KEY` — your Azure AI Search query or admin API key
- Optional: `AZURE_SEARCH_VECTOR_FIELD_NAME` — the vector field to query if the index has more than one vector field
- A vector-enabled Azure AI Search index for true hybrid search
- Azure OpenAI deployment configured via environment variables
- Azure CLI authenticated (`az login`)

This notebook uses `VectorizableTextQuery`, which expects the Azure AI Search index to support query-time vectorization. If the configured index is text-only, the notebook explains what is missing instead of silently falling back to keyword search.

In [1]:
%pip install agent-framework azure-ai-projects azure-identity azure-search-documents python-dotenv -q

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.core.credentials import AzureKeyCredential
from azure.identity import DefaultAzureCredential
from azure.search.documents import SearchClient
from azure.search.documents.indexes import SearchIndexClient
from azure.search.documents.models import VectorizableTextQuery

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")
search_endpoint = os.getenv("AZURE_SEARCH_SERVICE_ENDPOINT")
search_index_name = os.getenv("AZURE_SEARCH_INDEX_NAME", "demo-datasource-ks-index")
search_api_key = os.getenv("AZURE_SEARCH_API_KEY")
vector_field_override = os.getenv("AZURE_SEARCH_VECTOR_FIELD_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name,
    "AZURE_SEARCH_SERVICE_ENDPOINT": search_endpoint,
    "AZURE_SEARCH_API_KEY": search_api_key,
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [13]:
# Create the Azure AI Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

In [4]:
search_credential = AzureKeyCredential(search_api_key)

search_client = SearchClient(
    endpoint=search_endpoint,
    index_name=search_index_name,
    credential=search_credential,
)

index_client = SearchIndexClient(
    endpoint=search_endpoint,
    credential=search_credential,
)

search_index = index_client.get_index(search_index_name)
vector_field_names = [
    field.name
    for field in search_index.fields
    if getattr(field, "vector_search_dimensions", None)
]

if vector_field_override:
    vector_field_name = vector_field_override
elif vector_field_names:
    vector_field_name = vector_field_names[0]
else:
    vector_field_name = None

print(f"Azure AI Search index configured: {search_index_name}")
if vector_field_name:
    print(f"Hybrid search vector field: {vector_field_name}")
else:
    print(
        "This index does not expose a vector field. "
        "The hybrid-search tool below is implemented, but it requires a vector-enabled index to run."
    )

Azure AI Search index configured: demo-datasource-ks-index
This index does not expose a vector field. The hybrid-search tool below is implemented, but it requires a vector-enabled index to run.


In [15]:
search_client

<SearchClient [endpoint='https://azure-search-service-01.search.windows.net', index='demo-datasource-ks-index']>

## What is Agentic RAG?

Traditional RAG follows a fixed pipeline: retrieve documents, then generate a response. **Agentic RAG** goes further by giving the agent autonomy to decide **when** and **how** to retrieve information.

With Agentic RAG, the agent can:
- **Decide** whether retrieval is needed before answering a question
- **Choose** which data source or tool to query
- **Evaluate** retrieved results and perform follow-up retrievals if the first attempt is insufficient
- **Combine** information from multiple retrieval steps into a coherent answer

This makes the agent more flexible and accurate compared to a static retrieve-then-generate pipeline.

## Creating a Hybrid Search Tool

In Agentic RAG, external data sources are wrapped as **tools** that the agent can invoke on demand. This lets the agent treat retrieval as just another action it can take, rather than a mandatory step.

Below we expose your Azure AI Search index as a hybrid retrieval tool. Hybrid search combines keyword matching (`search_text`) with vector similarity (`vector_queries`) so exact terms like character names and semantic concepts can both influence ranking.

Hybrid search requires a vector-enabled index. If the configured index does not contain a vector field, the notebook shows a setup message instead of pretending the query is hybrid.

In [ ]:
def format_search_result(result: dict) -> str:
    visible_fields = []

    for field_name, field_value in result.items():
        if field_name.startswith("@search."):
            continue
        if field_value is None:
            continue
        if field_name.lower().endswith("vector"):
            continue

        field_text = " ".join(str(field_value).split())
        if len(field_text) > 500:
            field_text = f"{field_text[:500]}..."
        visible_fields.append(f"{field_name}: {field_text}")

    return "\n".join(visible_fields)


def run_hybrid_search(query: str, top: int = 3):
    if not vector_field_name:
        raise ValueError(
            f"Index '{search_index_name}' does not contain a vector field. "
            "Hybrid search requires both searchable text fields and a vector field with embeddings."
        )

    vector_query = VectorizableTextQuery(
        text=query,
        k_nearest_neighbors=top,
        fields=vector_field_name,
    )

    return search_client.search(
        search_text=query,
        vector_queries=[vector_query],
        top=top,
    )


# the hybrid search is a tool in Agentic RAG
@tool(approval_mode="never_require")
def search_knowledge_base(
    query: Annotated[str, "The search query for the Azure AI Search index"]
) -> str:
    """Search the Azure AI Search index using hybrid keyword and vector retrieval."""
    try:
        results = run_hybrid_search(query, top=3)
    except Exception as error:
        return f"Hybrid search is not available for this index configuration: {error}"

    matches = []
    for result in results:
        formatted_result = format_search_result(dict(result))
        if formatted_result:
            matches.append(formatted_result)

    return (
        "\n\n---\n\n".join(matches)
        if matches
        else "No matching documents found in the Azure AI Search index."
    )

In [6]:
if vector_field_name:
    sample_results = run_hybrid_search("identity and belonging in Barbie Land", top=1)

    for result in sample_results:
        print(format_search_result(dict(result)))
else:
    print(
        "Hybrid search is implemented, but this index is text-only. "
        "Use a vector-enabled Azure AI Search index, or set AZURE_SEARCH_INDEX_NAME and "
        "AZURE_SEARCH_VECTOR_FIELD_NAME for an index that contains embeddings."
    )

Hybrid search is implemented, but this index is text-only. Use a vector-enabled Azure AI Search index, or set AZURE_SEARCH_INDEX_NAME and AZURE_SEARCH_VECTOR_FIELD_NAME for an index that contains embeddings.


## Building the RAG Agent

Now we create an agent that is instructed to **always retrieve information before answering**. The agent uses the `search_knowledge_base` tool to ground its responses in your Azure AI Search index. In this hybrid version, the tool sends both the original text query and a vector query to Azure AI Search.

In [20]:
agent = client.as_agent(
    tools=[search_knowledge_base],
    name="AzureSearchRAGAgent",
    instructions="""You are a knowledgeable assistant. Before answering questions:
1. ALWAYS search the Azure AI Search index first
2. Base your answers on retrieved information
3. If information is not in the index, say so clearly
4. Cite concrete details from the retrieved snippets.""",
)

response = await agent.run(
    "What does the knowledge base say about Barbie Land ?",
)
print(response)

The knowledge base contains references to "Barbie Land" as a distinct location in the context of a screenplay or story involving Barbie characters. One snippet mentions that some characters rollerbladed to Barbie Land and that bringing humans there could lead to extremely weird things for the real world. Another snippet reflects on how Barbie Land was "ruined" by drawings described as weird, dark, and crazy. It appears that Barbie Land is a fictional place tied to the Barbie universe with a mix of whimsical and strange elements.

For detailed context, the information is derived from a screenplay document titled "BARBIE FINAL 2023 GG SCREENPLAY," which includes scenes and dialogue related to Barbie Land.

If you want more detailed or specific information about Barbie Land, please let me know!


## Iterative Retrieval — The Maker-Checker Pattern

A key advantage of Agentic RAG is **iterative retrieval**. The agent can perform multiple rounds of search to verify, refine, or expand on its initial findings — similar to a "maker-checker" workflow:

1. **Maker step**: The agent retrieves initial information and drafts a response.
2. **Checker step**: The agent performs additional retrievals to verify details or fill gaps.

Below, the agent is asked a question that requires comparing information from multiple retrieval results, prompting it to search several times.

In [21]:
checker_agent = client.as_agent(
    tools=[search_knowledge_base],
    name="AzureSearchRAGCheckerAgent",
    instructions="""You are a meticulous screenplay knowledge-base assistant who verifies answers against retrieved document snippets.
When answering questions about characters, scenes, dialogue, or events:
1. Search the Azure AI Search index for the main names, titles, or phrases in the question
2. Search again using specific character names, scene details, or quoted phrases found in the first results
3. Compare the retrieved snippets for consistent evidence about actions, relationships, and context
4. Ground the final answer in the retrieved snippets and mention the source document or blob URL when available
5. If the index does not contain enough evidence, clearly say what is missing instead of guessing.""",
)

response = await checker_agent.run(
    "Compare what the knowledge base says about Barbie and Ken.",
)
print(response)

The knowledge base provides several insights about Barbie and Ken from the screenplay "Barbie FINAL 2023 GG":

Barbie:
- Barbie is portrayed by Margot Robbie.
- She experiences complex emotions such as anxiety and fear without a specific object, showing vulnerability and depth. For example, Barbie mentions "I’ve started to get all these weirdo FEELINGS. Ugh. Like I have fear with no specific object," and this is identified as anxiety by a character passing by.
- Barbie has interactions that reflect her inner struggles, such as a scene where she discusses her "feet" issue with Weird Barbie.
- Barbie seems to be searching for someone important, indicated by "She’s got to be here somewhere..." which shows her purposeful and caring nature.
- At one point, Barbie faces a difficult emotional low, described as "This is the lowest I’ve ever been. Emotionally AND physically."

Ken:
- Ken is portrayed by Ryan Gosling.
- Ken appears confident and upbeat, as reflected in his statement "I feel amaz

## Summary

In this lesson you learned how to build an **Agentic RAG** system using the Microsoft Agent Framework and Azure AI Search hybrid retrieval:

- **Agentic RAG** lets agents autonomously decide when to retrieve information, making retrieval dynamic rather than fixed.
- **Hybrid search** combines keyword matching with vector similarity so exact terms and semantic meaning can both influence retrieval.
- **Tools as data sources**: Azure AI Search retrieval is wrapped as a tool the agent can invoke.
- **Iterative retrieval**: The maker-checker pattern enables the agent to perform multiple retrieval rounds — searching, verifying, and refining — before producing a final answer.

True hybrid search requires a vector-enabled Azure AI Search index. If the configured index is text-only, the notebook reports that clearly so learners can distinguish keyword retrieval from hybrid retrieval.